In [ ]:
# pip install arxiv
import importlib
import json

arxiv = importlib.import_module("arxiv")

client = arxiv.Client(page_size=200, delay_seconds=3, num_retries=5)
queries = [
    'abs:"retrieval augmented generation"',
    'abs:"dense retrieval"',
    'abs:"semantic search"',
    'abs:"large language model" AND abs:retrieval',
]

docs = {}
for q in queries:
    search = arxiv.Search(
        query=f'({q}) AND (cat:cs.CL OR cat:cs.IR OR cat:cs.LG)',
        max_results=3000,
        sort_by=arxiv.SortCriterion.Relevance,
    )
    for r in client.results(search):
        docs[r.entry_id] = {
            "id": r.entry_id,
            "title": r.title,
            "abstract": r.summary.replace("\n", " "),
            "year": r.published.year,
        }

with open("corpus.jsonl", "w") as f:
    for d in docs.values():
        f.write(json.dumps(d) + "\n")
        
print(len(docs))

In [1]:
import pandas as pd

df = pd.read_json("corpus.jsonl", lines=True)
df["n_words"] = df["abstract"].str.split().str.len()

print(df.shape)
print(df["n_words"].describe())
print(df["year"].value_counts().sort_index())
print("empty abstracts:", (df["abstract"].str.strip() == "").sum())
print("duplicate titles:", df["title"].str.lower().duplicated().sum())

print(df.sample(3, random_state=0)[["title", "abstract"]].to_string())

(5648, 5)
count    5648.000000
mean      169.970786
std        38.292621
min        36.000000
25%       144.000000
50%       168.000000
75%       193.000000
max       313.000000
Name: n_words, dtype: float64
year
2009       1
2010       1
2011       1
2012       6
2013       5
2014       6
2015       1
2016       5
2017       7
2018      17
2019      11
2020      20
2021      73
2022     133
2023     386
2024    1323
2025    2089
2026    1563
Name: count, dtype: int64
empty abstracts: 0
duplicate titles: 2
                                                                                                                        title                                                                                                                                                                                                                                                                                                                                                                           

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_json("corpus.jsonl", lines=True)
df = df.drop_duplicates(subset="title", keep="first").reset_index(drop=True)
df["text"] = df["title"] + ". " + df["abstract"]   # title helps retrieval
df.to_json("corpus_clean.jsonl", orient="records", lines=True)

print(df.nsmallest(3, "n_words")[["title", "abstract"]] if "n_words" in df else
      df.assign(n=df.abstract.str.split().str.len()).nsmallest(3, "n")[["title", "abstract"]])

# Baseline config: decide now, then freeze it
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True,
                      ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(df["text"])
print(X.shape)

def search_tfidf(query, k=5):
    q = vec.transform([query])
    scores = cosine_similarity(q, X).ravel()
    top = scores.argsort()[::-1][:k]
    return [(df.title[i], round(float(scores[i]), 3)) for i in top]

for q in ["how do models look things up instead of memorizing facts",
          "reducing made-up answers from language models"]:
    print("\n", q)
    for t, s in search_tfidf(q):
        print(f"  {s}  {t}")

                                                  title  \
30    Towards Comprehensive Vietnamese Retrieval-Aug...   
15    Observations on Building RAG Systems for Techn...   
3127  Dense Retrieval for Low Resource Languages -- ...   

                                               abstract  
30    This paper presents our contributions towards ...  
15    Retrieval augmented generation (RAG) for techn...  
3127  This paper reports some difficulties and some ...  
(5646, 81521)

 how do models look things up instead of memorizing facts
  0.102  Generative AI for Low-Carbon Artificial Intelligence of Things with Large Language Models
  0.098  Generative Dense Retrieval: Memory Can Be a Burden
  0.094  Provable Benefits of In-Tool Learning for Large Language Models
  0.082  Generative Cross-Modal Retrieval: Memorizing Images in Multimodal Language Models for Retrieval and Beyond
  0.068  Semantic Advertising

 reducing made-up answers from language models
  0.105  FilterRAG: Zero-Shot In

In [3]:
import json

queries = {
    "q1": "how do models look things up instead of memorizing facts",
    "q2": "splitting long documents into pieces before indexing",
    "q3": "retrieval for languages with little training data",
    "q4": "making retrieval faster and cheaper at scale",
    "q5": "tell me about RAG",
    "q6": "what is RAG in machine learning",
    "q7": "where is RAG used nowadays",
    "q8": "how do you evaluate RAG systems",
    "q9": "efficiency and performance of these",
    "q10": "improving retrieval for long documents",
    "q11": "handling ambiguous queries",
    "q12": "handling multiple answers",
    "q13": "coding help using these",
    "q14": "how is RAG different from transformers",
    "q15": "is RAG used in ChatGPT",
    "q16": "current problems with RAG",
    "q18": "intro to RAG",
    "q19": "brief me on this topic in about 50 words",
    "q20": "what is the main idea behind RAG",
    "q21": "rag is used in LLMs",
    "q22": "rag used in small & simple chatbots",
    "q23": "rag used in large & complex chatbots",\
    "q24": "rag future",
    "q25": "rag current developments",
    "q26": "rag comparison with time",
    "q27": "rag comparison with other methods",
    "q28": "handling conflicting retrieved sources",
    "q29": "privacy in rag",
    
}
json.dump(queries, open("queries.json", "w"), indent=2)

In [4]:
import os
print(os.path.exists("queries.json"))   # should print True

True


In [5]:
import json, os
import numpy as np

LABELS = "labels.json"   # {qid: {doc_idx: 0/1/2}}
labels = json.load(open(LABELS)) if os.path.exists(LABELS) else {}

if os.path.exists("queries.json"):
    queries = json.load(open("queries.json"))
else:
    queries = {
        "q1": "how do models look things up instead of memorizing facts",
        "q2": "reducing made-up answers from language models",
    }
    print("queries.json not found; using built-in evaluation queries.")

def label_pool(system_fn, k=10):
    """system_fn(query, k) -> list of doc indices. Judge unseen docs only."""
    for qid, q in queries.items():
        seen = labels.setdefault(qid, {})
        for i in system_fn(q, k):
            if str(i) in seen:
                continue
            print(f"\nQ: {q}\n[{i}] {df.title[i]}\n{df.abstract[i][:500]}")
            seen[str(i)] = int(input("0=no 1=partial 2=yes > "))
            json.dump(labels, open(LABELS, "w"))

def mrr_ndcg(system_fn, k=10):
    rr, nd = [], []
    for qid, q in queries.items():
        rel = labels.get(qid, {})
        ranked = system_fn(q, k)
        gains = [rel.get(str(i), 0) for i in ranked]
        first = next((r for r, g in enumerate(gains, 1) if g >= 1), None)
        rr.append(1 / first if first else 0)
        dcg = sum((2**g - 1) / np.log2(r + 1) for r, g in enumerate(gains, 1))
        ideal = sorted(rel.values(), reverse=True)[:k]
        idcg = sum((2**g - 1) / np.log2(r + 1) for r, g in enumerate(ideal, 1))
        nd.append(dcg / idcg if idcg else 0)
    return np.mean(rr), np.mean(nd), rr, nd

# TF-IDF returning indices
def tfidf_idx(q, k=10):
    s = cosine_similarity(vec.transform([q]), X).ravel()
    return s.argsort()[::-1][:k].tolist()

In [16]:
print(len(queries))

28


In [6]:
label_pool(tfidf_idx, k=10)

## Phase 2 begins here...

In [7]:
print(len(queries))                   
print(mrr_ndcg(tfidf_idx, k=10))

28
(np.float64(0.894047619047619), np.float64(0.4912594111587885), [0.5, 1.0, 0.5, 1.0, 0.2, 0.3333333333333333, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [np.float64(0.09975026728232968), np.float64(0.6000154988728346), np.float64(0.5474322129301818), np.float64(0.6870072382962691), np.float64(0.11106055368337021), np.float64(0.052787337918620024), np.float64(0.38744638578218565), np.float64(0.8036197760185034), np.float64(0.5091167764432026), np.float64(0.7141111399395731), np.float64(0.360827234459509), np.float64(0.3661481637523776), np.float64(0.3990867852425513), np.float64(0.2530629685756193), np.float64(0.12770807883519136), np.float64(0.5343667329673865), np.float64(0.39927349988139943), np.float64(0.37643289080104797), np.float64(0.26708117509254947), np.float64(0.8709540216078537), np.float64(0.7804414695846056), np.float64(0.6372563248426917), np.float64(0.7103147983233091), np.float64(0.6984028435244843),

In [8]:
mrr, ndcg, rr_tfidf, nd_tfidf = mrr_ndcg(tfidf_idx)
print(mrr, ndcg)

# worst queries: where TF-IDF fails most
for qid, s in sorted(zip(queries, nd_tfidf), key=lambda x: x[1])[:5]:
    print(round(s, 3), queries[qid])

0.894047619047619 0.4912594111587885
0.053 what is RAG in machine learning
0.1 how do models look things up instead of memorizing facts
0.111 tell me about RAG
0.128 is RAG used in ChatGPT
0.253 how is RAG different from transformers


In [ ]:
for qid, q in queries.items():
    pos = sum(g >= 1 for g in labels.get(qid, {}).values())
    if pos < 2:
        print(qid, pos, q)

#Outputs q17 has 0 documents labelled 1 or 2. So query 17 was dropped.

In [9]:
drop = ["what is RAG in machine learning", "tell me about RAG",
        "what is the main idea behind RAG"]
# These queries perform poorly and have few positive labels, so we drop them from the queries-set.
queries = {qid: q for qid, q in queries.items() if q not in drop}
json.dump(queries, open("queries.json", "w"), indent=2)
print(len(queries))

25


In [10]:
df = pd.read_json("corpus_clean.jsonl", lines=True)
print(df.shape, "text" in df.columns)   # expect (5646, ...) True

(5646, 5) True


In [11]:
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True,
                      ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(df["text"])

In [12]:
label_pool(tfidf_idx, k=10)
mrr, ndcg, rr_tfidf, nd_tfidf = mrr_ndcg(tfidf_idx)
print(len(queries), mrr, ndcg)
json.dump({"mrr": mrr, "ndcg": ndcg, "per_query": dict(zip(queries, nd_tfidf))},
          open("phase1_results.json", "w"))

25 0.94 0.5329733778300616


### Averaged GloVe Vectors

In [13]:
# pip install gensim
%pip install -q "gensim>=4.3.3"

import re
import gensim.downloader as api

kv = api.load("glove-wiki-gigaword-300")  # ~400 MB, one-time download

# IDF weights from a unigram vectorizer
uni = TfidfVectorizer(stop_words="english", min_df=2).fit(df["text"])
idf = dict(zip(uni.get_feature_names_out(), uni.idf_))
idf_default = max(idf.values())
stop_words = uni.get_stop_words() or set()

def embed(text):
    tokens = [
        token for token in re.findall(r"[a-z]+", text.lower())
        if token in kv and token not in stop_words
    ]

    if not tokens:
        return np.zeros(kv.vector_size, dtype=np.float32)

    weights = np.array(
        [idf.get(token, idf_default) for token in tokens],
        dtype=np.float32,
    )
    vectors = np.array([kv[token] for token in tokens])
    vector = (vectors * weights[:, None]).sum(axis=0) / weights.sum()

    norm = np.linalg.norm(vector)
    return vector / norm if norm else vector

D = np.vstack([embed(text) for text in df["text"]])

def glove_idx(q, k=10):
    return (D @ embed(q)).argsort()[::-1][:k].tolist()

for q in [
    "how do models look things up instead of memorizing facts",
    "reducing made-up answers from language models",
]:
    print("\n", q)
    for i in glove_idx(q, 5):
        print("  ", df.title.iloc[i])



Note: you may need to restart the kernel to use updated packages.

 how do models look things up instead of memorizing facts
   SuRe: Summarizing Retrievals using Answer Candidates for Open-domain QA of LLMs
   Good/Evil Reputation Judgment of Celebrities by LLMs via Retrieval Augmented Generation
   Insights into LLM Long-Context Failures: When Transformers Know but Don't Tell
   Large Language Models and Multimodal Retrieval for Visual Word Sense Disambiguation
   Thought-Retriever: Don't Just Retrieve Raw Data, Retrieve Thoughts for Memory-Augmented Agentic Systems

 reducing made-up answers from language models
   Augmenting LLM Reasoning with Dynamic Notes Writing for Complex QA
   Ask Good Questions for Large Language Models
   Retrieval Augmented Question Answering: When Should LLMs Admit Ignorance?
   Adapting Standard Retrieval Benchmarks to Evaluate Generated Answers
   Learning by Analogy: Enhancing Few-Shot Prompting for Math Word Problem Solving with Computational Graph-Ba

In [15]:
import os
print(os.path.exists("glove_doc_vectors.npy"))
np.save("glove_doc_vectors.npy", D)

True


In [16]:
label_pool(glove_idx, k=10)
print(mrr_ndcg(glove_idx)[:2])
print(mrr_ndcg(tfidf_idx)[:2])

(np.float64(0.9), np.float64(0.4824407768451543))
(np.float64(0.94), np.float64(0.5329733778300616))


In [17]:
# 1. confirm labeling is complete: this should print nothing and ask nothing
label_pool(glove_idx, k=10)

# 2. scores against the now-larger pool
m_t, n_t, rr_t, nd_t = mrr_ndcg(tfidf_idx)
m_g, n_g, rr_g, nd_g = mrr_ndcg(glove_idx)
print(f"TF-IDF  MRR {m_t:.3f}  NDCG@10 {n_t:.3f}")
print(f"GloVe   MRR {m_g:.3f}  NDCG@10 {n_g:.3f}")

# 3. per-query comparison
wins = sum(g > t for g, t in zip(nd_g, nd_t))
ties = sum(g == t for g, t in zip(nd_g, nd_t))
print(f"GloVe wins {wins}, ties {ties}, loses {len(nd_t) - wins - ties} of {len(nd_t)}")

# biggest gaps in each direction
diff = sorted(zip(queries.values(), nd_g, nd_t), key=lambda x: x[1] - x[2])
print("\nGloVe worst vs TF-IDF:")
for q, g, t in diff[:4]:
    print(f"  {g - t:+.2f}  {q}")
print("\nGloVe best vs TF-IDF:")
for q, g, t in diff[-4:]:
    print(f"  {g - t:+.2f}  {q}")

# 4. save
json.dump({"tfidf": {"mrr": m_t, "ndcg": n_t, "per_query": dict(zip(queries, nd_t))},
           "glove": {"mrr": m_g, "ndcg": n_g, "per_query": dict(zip(queries, nd_g))}},
          open("results_phase2.json", "w"))

TF-IDF  MRR 0.940  NDCG@10 0.533
GloVe   MRR 0.900  NDCG@10 0.482
GloVe wins 12, ties 0, loses 13 of 25

GloVe worst vs TF-IDF:
  -0.58  rag used in small & simple chatbots
  -0.54  rag is used in LLMs
  -0.48  rag used in large & complex chatbots
  -0.36  intro to RAG

GloVe best vs TF-IDF:
  +0.23  privacy in rag
  +0.27  how do models look things up instead of memorizing facts
  +0.33  is RAG used in ChatGPT
  +0.37  rag comparison with other methods


In [18]:
print("rag" in kv, "llms" in kv, "llm" in kv)
print(kv.most_similar("rag", topn=5))

True False True
[('rags', 0.5144805312156677), ('doll', 0.44610339403152466), ('towel', 0.43989768624305725), ('cloth', 0.416219025850296), ('soaked', 0.40774789452552795)]


In [35]:
print(len(queries)); print(list(queries.values()))

28
['how do models look things up instead of memorizing facts', 'splitting long documents into pieces before indexing', 'retrieval for languages with little training data', 'making retrieval faster and cheaper at scale', 'tell me about RAG', 'what is RAG in machine learning', 'where is RAG used nowadays', 'how do you evaluate RAG systems', 'efficiency and performance of these', 'improving retrieval for long documents', 'handling ambiguous queries', 'handling multiple answers', 'coding help using these', 'how is RAG different from transformers', 'is RAG used in ChatGPT', 'current problems with RAG', 'intro to RAG', 'brief me on this topic in about 50 words', 'what is the main idea behind RAG', 'rag is used in LLMs', 'rag used in small & simple chatbots', 'rag used in large & complex chatbots', 'rag future', 'rag current developments', 'rag comparison with time', 'rag comparison with other methods', 'handling conflicting retrieved sources', 'privacy in rag']


In [1]:
%pip install -q --upgrade sentence-transformers sentence-transformers

^C
Note: you may need to restart the kernel to use updated packages.


In [19]:
from sentence_transformers import SentenceTransformer
import numpy as np, os

model = SentenceTransformer("all-MiniLM-L6-v2")

if os.path.exists("st_doc_vectors.npy"):
    E = np.load("st_doc_vectors.npy")
else:
    E = model.encode(df["text"].tolist(), batch_size=64,
                     normalize_embeddings=True, show_progress_bar=True)
    np.save("st_doc_vectors.npy", E)

def st_idx(q, k=10):
    qv = model.encode([q], normalize_embeddings=True)[0]
    return (E @ qv).argsort()[::-1][:k].tolist()

d:\My Projects\RAGger\RAGvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].

In [21]:
print(E.shape)   # expect (5646, 384)
for q in ["how do models look things up instead of memorizing facts",
          "reducing made-up answers from language models"]:
    print("\n", q)
    for i in st_idx(q, 5):
        print("  ", df.title.iloc[i])

(5646, 384)

 how do models look things up instead of memorizing facts
   Disentangling Memory and Reasoning Ability in Large Language Models
   Provable Benefits of In-Tool Learning for Large Language Models
   Rethinking Memory in LLM based Agents: Representations, Operations, and Emerging Topics
   LLM In-Context Recall is Prompt Dependent
   Do LLMs dream of elephants (when told not to)? Latent concept association and associative memory in transformers

 reducing made-up answers from language models
   Decomposing and Revising What Language Models Generate
   Retrieving Supporting Evidence for LLMs Generated Answers
   RAG-QA Arena: Evaluating Domain Robustness for Long-form Retrieval Augmented Question Answering
   KS-LLM: Knowledge Selection of Large Language Models with Evidence Document for Question Answering
   Measuring the Quality of Answers in Political Q&As with Large Language Models


In [22]:
label_pool(st_idx, k=10)    # only unjudged docs appear

systems = {"tfidf": tfidf_idx, "glove": glove_idx, "minilm": st_idx}
res = {}
for name, fn in systems.items():
    m, n, rr, nd = mrr_ndcg(fn)
    res[name] = {"mrr": m, "ndcg": n, "rr": rr, "nd": nd}
    print(f"{name:7s} MRR {m:.3f}  NDCG@10 {n:.3f}")

# minilm vs each baseline, per query
for base in ["tfidf", "glove"]:
    d = np.array(res["minilm"]["nd"]) - np.array(res[base]["nd"])
    print(f"minilm vs {base}: wins {(d > 0).sum()}, ties {(d == 0).sum()}, loses {(d < 0).sum()}")

# where MiniLM is weakest vs the best baseline
best_base = np.maximum(res["tfidf"]["nd"], res["glove"]["nd"])
gap = sorted(zip(queries.values(), np.array(res["minilm"]["nd"]) - best_base),
             key=lambda x: x[1])
print("\nMiniLM worst vs best baseline:")
for q, g in gap[:5]:
    print(f"  {g:+.2f}  {q}")

json.dump({n: {"mrr": r["mrr"], "ndcg": r["ndcg"],
               "per_query": dict(zip(queries, r["nd"]))} for n, r in res.items()},
          open("results_phase3.json", "w"))


Q: current problems with RAG
[733] Cost-Efficient RAG for Entity Matching with LLMs: A Blocking-based Exploration
Retrieval-augmented generation (RAG) enhances LLM reasoning in knowledge-intensive tasks, but existing RAG pipelines incur substantial retrieval and generation overhead when applied to large-scale entity matching. To address this limitation, we introduce CE-RAG4EM, a cost-efficient RAG architecture that reduces computation through blocking-based batch retrieval and generation. We also present a unified framework for analyzing and evaluating RAG systems for entity matching, focusing on blocking-a

Q: intro to RAG
[943] RAGChecker: A Fine-grained Framework for Diagnosing Retrieval-Augmented Generation
Despite Retrieval-Augmented Generation (RAG) showing promising capability in leveraging external knowledge, a comprehensive evaluation of RAG systems is still challenging due to the modular nature of RAG, evaluation of long-form responses and reliability of measurements. In thi